In [13]:
from pyomo.environ import *

# Create a ConcreteModel
model = ConcreteModel()

# Define sets for rows and columns
model.Rows = RangeSet(0, 4)
model.Cols = RangeSet(0, 4)

# Define binary decision variables
model.x = Var(model.Rows, model.Cols, within=Binary)
model.n = Var(model.Rows, model.Cols, within=Binary)

# Define objective function
model.obj = Objective(expr=sum(model.x[i, j] for i in model.Rows for j in model.Cols), sense=minimize)

# Define constraints
model.constraints = ConstraintList()
for i in model.Rows:
    for j in model.Cols:
        # Bulb must be either turned off directly or indirectly by touching
        model.constraints.add(model.x[i, j] >= model.n[i, j])
        
        # If a bulb is touched, at least one of its neighbors must also be touched
        if i > 0:
            model.constraints.add(model.n[i, j] <= model.x[i-1, j] + model.n[i-1, j])
        if i < 4:
            model.constraints.add(model.n[i, j] <= model.x[i+1, j] + model.n[i+1, j])
        if j > 0:
            model.constraints.add(model.n[i, j] <= model.x[i, j-1] + model.n[i, j-1])
        if j < 4:
            model.constraints.add(model.n[i, j] <= model.x[i, j+1] + model.n[i, j+1])
        
        # For bulbs in the edges, adjust the neighbor count accordingly
        if i == 0 and 0 < j < 4:
            model.constraints.add(model.n[i, j] <= sum(model.x[i+di, j+dj] for di, dj in ((1, 0), (0, 1), (0, -1))))
        elif i == 4 and 0 < j < 4:
            model.constraints.add(model.n[i, j] <= sum(model.x[i+di, j+dj] for di, dj in ((-1, 0), (0, 1), (0, -1))))
        elif j == 0 and 0 < i < 4:
            model.constraints.add(model.n[i, j] <= sum(model.x[i+di, j+dj] for di, dj in ((1, 0), (-1, 0), (0, 1))))
        elif j == 4 and 0 < i < 4:
            model.constraints.add(model.n[i, j] <= sum(model.x[i+di, j+dj] for di, dj in ((1, 0), (-1, 0), (0, -1))))
        
        # For bulbs in the corners, adjust the neighbor count accordingly
        if (i == 0 and j == 0) or (i == 0 and j == 4) or (i == 4 and j == 0) or (i == 4 and j == 4):
            model.constraints.add(model.n[i, j] <= sum(model.x[i+di, j+dj] for di, dj in ((1, 0), (-1, 0), (0, 1), (0, -1))))

# Solve the problem
solver = SolverFactory('gurobi')
solver.solve(model)

# Print the minimum number of bulbs to switch off
print("Minimum number of bulbs to switch off:", sum(value(model.x[i, j]) for i in model.Rows for j in model.Cols))

# Print the positions of the bulbs that need to be switched off
print("Bulbs to switch off:")
for i in model.Rows:
    for j in model.Cols:
        if value(model.x[i, j]) == 1:
            print(f"Bulb at ({i+1},{j+1})")

KeyError: "Index '(-1, 0)' is not valid for indexed component 'x'"